###Reference

For this project, the main tutorial I'll be using could be accessed through this link: https://medium.com/@o39joey/introduction-to-rag-with-python-langchain-62beeb5719ad. The information in this site will be used to create the basic RAG, but I'll add changes as I see fit.

In [1]:
#installs important depedencies for this project
#chroma is the used vector database
#langchain is a framework for developing applications that use LLMs
#openai will be the LLM that's used
def download_dependencies():
  dependencies = [
      "pypdf",
      "langchain-chroma",
      "langchain-core",
      "langchain-community",
      "langchain-google-genai",
      "langchain-google-gemini",
      "langchain-text-splitters",
  ]


  for dependency in dependencies:
    !pip install "{dependency}"

In [10]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 8.1 MB/s eta 0:00:00


In [2]:
download_dependencies()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

###Choice of LLM

For this project, I'll use gemini because it is cost effective and has good performance.

In [3]:
#loading in gemini api key
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API')

###Embedder Function

This function loads the documents, creates them, and then stores them in the vector database.

In [16]:
#function that embeds text
def embed_text(loader, splitter, vector_store):
  docs = loader.load()
  texts = splitter.split_documents(docs)
  return vector_store.add_documents(documents=texts)


###Injecting Dependencies and Instantiating the Class

Because BrightBridge is an application that focuses on mental and relationship health, a good source to index would be "Intimate Relationships (9th Edition)."

Since I have it as a pdf, I'll be using Langchain's pdf loader.

The pdf content will also be split into 1000 character chunks to be individually embedded.

This initializes a google gemini embedder that converts the text into vector embeddings. Afterwards, a vector database (in this case, ChromaDB) is initialized. [0, 1]

In [34]:
# imports
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import userdata

# Parses the PDF
loader = PyPDFLoader("/content/Intimate Relationships (9th Edition with Contents)-555-577-10-23.pdf")

# Initialize Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

# Get Embeddings Model
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2",
                                          google_api_key=GEMINI_API_KEY)

#connects the Chroma object to the cloud instance
vector_store = Chroma(
    collection_name="intimate_relationships_collection",
    embedding_function=embeddings,
    chroma_cloud_api_key=userdata.get("CHROMA_API_KEY"),
    tenant=userdata.get("CHROMA_TENANT"),
    database="Demo",
)

In [35]:
#embed text into the database
ids = embed_text(loader, text_splitter, vector_store)

###Testing the VectorDB



In [ ]:
# Query the Vector Store
results = vector_store.similarity_search(
    'What are some therapy options?',
    k=2
)

# Print Resulting Chunks
for res in results:
    print(f"* {res.page_content} [{res.metadata}]\n\n")

###Naive RAG Implementation

Here we will start implementing our naive RAG. [2, 3]

In [19]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

class Naive_RAG:
  #initializes naive rage with the gemini llm and the chromadb retriever
  def __init__(self, llm, retriever, prompt):

    # Create Prompt Instance from template
    custom_rag_prompt = PromptTemplate.from_template(prompt)

    self.rag_chain = (
    {"context": retriever | self.format_docs, "query": RunnablePassthrough()}
    | custom_rag_prompt
    | llm
    | StrOutputParser()
    )

  # Create Document Parsing Function to String
  def format_docs(self, docs):
      return "\n\n".join(doc.page_content for doc in docs)

  #invokes the prompt
  def invoke(self, prompt):
    return self.rag_chain.invoke(prompt)

In [24]:
from langchain_google_genai import ChatGoogleGenerativeAI

#gemini as llm
llm =  ChatGoogleGenerativeAI(model="gemini-2.5-flash",
                               api_key=GEMINI_API_KEY)

# Set Chroma Vector Store as the Retriever
retriever = vector_store.as_retriever()

# Create the Prompt Template
prompt_template = """Use the context provided to answer
the user's question below. However, if there is no context,
tell the user there is no verified information about their question,
and try to answer the question based on your own knowledge.

context: {context}

question: {query}

answer: """

rag = Naive_RAG(llm, retriever, prompt_template)

In [36]:
import textwrap

prompt = "what are some good therapy options?"

output = rag.invoke(prompt)
wrapped_text = textwrap.fill(output, width=60)
print(wrapped_text)

Based on the context provided, the text discusses different types of couple therapy without detailing many specific options. However, it does mention:

*   **IBCT (Integrative Behavioral Couple Therapy)**, with a version available online at www.OurRelationship.com.
*   **Emotionally-Focused-Couple-Therapy (EFCT)**, which is derived from attachment theory and strives to improve relationships.

The most important advice given in the text regarding choosing a therapy is: **"Pick the therapy—and the therapist—that appeal to you the most."** This is because, despite their different labels, the therapies all share common features and are generally effective. The text states that "they all work," and between 60 to 70 percent of couples who seriously undertake any of these therapies achieve notable reductions in dissatisfaction and distress. The therapist's credibility and persuasiveness are also highlighted as important factors for success.


###Langchain Documentation
0. https://docs.langchain.com/oss/python/integrations/embeddings/google_generative_ai

1.  Langchain Chromadb -- https://docs.langchain.com/oss/python/integrations/vectorstores/chroma

Langchain documentation w/ Gemini:

2.  https://docs.langchain.com/oss/python/langchain/rag#chroma
3.  https://reference.langchain.com/python/langchain-google-genai/chat_models/ChatGoogleGenerativeAI